In [1]:
import pandas as pd
import numpy as np
pd.set_option('display.max_columns', None)

In [2]:
train = pd.read_csv('./data/train_reviews.csv')
test = pd.read_csv('./data/test_reviews.csv')
negocios = pd.read_csv('./data/negocios.csv')
usuarios = pd.read_csv('./data/usuarios.csv')

C:\Users\Lluis\AppData\Local\Temp\ipykernel_4688\4243529203.py:4: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  usuarios = pd.read_csv('./data/usuarios.csv')


In [3]:
train

,review_id,user_id,business_id,stars,useful,funny,cool,text,date
0,ZZO43qKB-s65zplC8RfJqw,-1BSu2dt_rOAqllw9ZDXtA,smkZq4G1AOm4V6p3id5sww,5.0,0,0,0,Fantastic fresh food. The greek salad is amazi...,2016-09-30 15:49:32
1,vojXOF_VOgvuKD95gCO8_Q,xpe178ng_gj5X6HgqtOing,96_c_7twb7hYRZ9HHrq01g,1.0,2,0,1,Been a patient at Largo Med/Diagnostic Clinic ...,2020-12-09 14:39:51
2,KwxdbiseRlIRNzpgvyjY0Q,axbaerf2Fk92OB4b9_peVA,e0AYjKfSF0DL-5C1CpOq6Q,4.0,0,0,0,The location is convenient to my campus so I d...,2013-09-04 16:19:51
3,3mwoBcTy-2gMh0L91uaIeA,_GOiybb0rImYKJfwyxEaGg,vF-uptiQ34pVLHJKzPHUlA,5.0,0,0,0,I agree with all the other compliments posted ...,2019-03-02 12:24:14
4,XfWf7XsBWs3kYyYq7Ns1ZQ,ojWKg3B5pH3ncAsxun3kUw,X28XK71RuEXPapeyUOwNzg,5.0,10,4,7,"Wanting to help out the local economy, I thoug...",2020-04-23 18:26:29
...,...,...,...,...,...,...,...,...,...
967779,yhyJUBAJUG-9gEIQ7gXS_g,iZAAjkPZ0sopzfajcfdOUg,2PvPsZ3KRFCtHbQkNHvGpg,5.0,1,0,0,Ordered takeout and this place didn't disappoi...,2020-04-19 15:42:12
967780,5NxUmqweVTAZ_-FGJywTYw,AN8XscFSH1jctLSqAQ9-bA,-K0zTgGyxo-AeSkcV0IVaA,4.0,0,0,0,I took our annual managers meeting there for d...,2014-02-10 18:27:03
967781,-0rEInvO7q5cSI81NlH3-g,qtgODIPIsKsH2j3rItF9Tw,eZCt3doDaA-l4sg_OM67YQ,5.0,0,0,1,My favorite local coffee shop! Great drinks (i...,2019-07-24 14:19:52
967782,8icm1hiV87OZbDsyjQDLAQ,gu7nU1IM7U3lLwUbqLeiDQ,ww3YJXu5c18aGZXWmm00qg,5.0,0,0,0,We hit a quiet time here on a very busy weeken...,2011-11-08 06:38:05


In [4]:
import torch
from torch import nn
from transformers import AutoModel, AutoTokenizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from transformers import RobertaTokenizer, DistilBertTokenizer
from transformers import LongformerTokenizer, LongformerModel


In [5]:
# train = train.drop(columns = ['date'])
train_df, val_df = train_test_split(train, test_size=0.2, random_state=42)

# Load pre-trained model and tokenizer
model_name = "google/bigbird-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)


In [6]:
from datasets import Dataset
text_dataset = Dataset.from_pandas(train_df[['text','stars']])
text_dataset = text_dataset.rename_column('stars', 'label')
text_dataset

Dataset({
    features: ['text', 'label', '__index_level_0__'],
    num_rows: 774227
})

In [7]:
split = text_dataset.train_test_split(test_size=0.2, shuffle=True, seed=42)
train_dataset = split['train']
val_dataset = split['test']


In [8]:
def tokenize_function(examples, tokenizer, max_length):
    """Tokenize the texts"""
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=max_length,
    )


In [9]:
MAX_LENGTH = 1048

In [10]:
tokenized_train_dataset = train_dataset.map(
    lambda x: tokenize_function(x, tokenizer, max_length=MAX_LENGTH),
    batched=True,
    remove_columns=["text"],
)
tokenized_val_dataset = val_dataset.map(
    lambda x: tokenize_function(x, tokenizer, max_length=MAX_LENGTH),
    batched=True,
    remove_columns=["text"],
)

Map:   0%|          | 0/619381 [00:00<?, ? examples/s]

Map:   0%|          | 0/154846 [00:00<?, ? examples/s]

In [11]:
tokenized_val_dataset = tokenized_val_dataset.remove_columns(["__index_level_0__"])
tokenized_train_dataset = tokenized_train_dataset.remove_columns(["__index_level_0__"])

In [12]:
from transformers import AutoModelForSequenceClassification
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=1,
    problem_type="regression"
)

Some weights of BigBirdForSequenceClassification were not initialized from the model checkpoint at google/bigbird-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [28]:
from peft import get_peft_model, LoraConfig, TaskType

lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["query", "key", "value"],  # depende del modelo
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 1,033,729 || all params: 129,093,890 || trainable%: 0.8008


c:\Users\Lluis\anaconda3\envs\Env\Lib\site-packages\peft\mapping_func.py:73: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
c:\Users\Lluis\anaconda3\envs\Env\Lib\site-packages\peft\tuners\tuners_utils.py:167: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [29]:
print('ready')

ready


In [30]:
train_dataset

Dataset({
    features: ['text', 'label', '__index_level_0__'],
    num_rows: 619381
})

In [31]:
torch.cuda.empty_cache()
import gc
gc.collect()


1937

In [32]:
from transformers import Trainer, TrainingArguments
from transformers import EarlyStoppingCallback
max_steps = 10  # Adjust the number of steps to your requirement

training_args = TrainingArguments(
    output_dir="./bigbird-lora-regression",
    per_device_train_batch_size=2,
    max_steps =max_steps,  # Training will stop after these steps
    logging_steps=1,
    evaluation_strategy="steps",  # Evaluate every 'logging_steps' steps
    save_strategy="steps",  # Save model every 'logging_steps' steps
    save_steps=250,  # Save model checkpoint every 500 steps
    learning_rate=2e-4,
    weight_decay=0.01,
    load_best_model_at_end=True,  # Load the best model after training
)

# Step 2: EarlyStoppingCallback
early_stopping = EarlyStoppingCallback(early_stopping_patience=5)  # Stop after 3 evaluations without improvement

# Step 3: Trainer with early stopping
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_val_dataset,
    tokenizer=tokenizer,
    callbacks=[early_stopping]
)

c:\Users\Lluis\anaconda3\envs\Env\Lib\site-packages\transformers\training_args.py:1594: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
C:\Users\Lluis\AppData\Local\Temp\ipykernel_4688\2311073147.py:22: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
No label_names provided for model class `PeftModelForSequenceClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [33]:
trainer.train()

TypeError: BigBirdForSequenceClassification.forward() got an unexpected keyword argument 'num_items_in_batch'

In [19]:
tokenized_val_dataset

Dataset({
    features: ['label', 'input_ids', 'attention_mask'],
    num_rows: 154846
})

In [20]:
small_eval_dataset = tokenized_val_dataset.select(range(10))
predictions = trainer.predict(small_eval_dataset)


In [27]:
small_eval_dataset['label'], predictions

([4.0, 5.0, 3.0, 5.0, 5.0, 5.0, 5.0, 4.0, 5.0, 5.0],
 PredictionOutput(predictions=array([[0.05078263],
        [0.08877696],
        [0.01163155],
        [0.10706078],
        [0.06768922],
        [0.07801278],
        [0.11485418],
        [0.10873059],
        [0.02833294],
        [0.08902362]], dtype=float32), label_ids=array([4., 5., 3., 5., 5., 5., 5., 4., 5., 5.], dtype=float32), metrics={'test_loss': 20.89830780029297, 'test_runtime': 1.064, 'test_samples_per_second': 9.398, 'test_steps_per_second': 1.88}))

In [ ]:
test_tokenized_dataset = Dataset.from_pandas(test[['text']])
test_tokenized_dataset = test_tokenized_dataset.map(
    lambda x: tokenize_function(x, tokenizer, max_length=MAX_LENGTH),
    batched=True,
    remove_columns=["text"],
)
test_tokenized_dataset = test_tokenized_dataset.remove_columns(["__index_level_0__"])
test_tokenized_dataset = test_tokenized_dataset.rename_column("label", "labels")

In [ ]:
predictions = trainer.predict(test_tokenized_dataset)
predictions = predictions.predictions.flatten()

In [ ]:
predictions[0:10]

In [34]:
model_name="distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

c:\Users\Lluis\anaconda3\envs\Env\Lib\site-packages\huggingface_hub\file_download.py:140: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Lluis\.cache\huggingface\hub\models--distilbert-base-uncased-finetuned-sst-2-english. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

In [35]:
text = train_dataset[0]['text']
inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

# Get the model's prediction
with torch.no_grad():
    outputs = model(**inputs)
    scores = outputs.logits.squeeze()
    
# Convert to probabilities
probs = torch.nn.functional.softmax(scores, dim=0)

In [36]:
probs

tensor([0.9621, 0.0379])

In [39]:
if len(probs) == 2:
# SST-2 has label 0 for negative and 1 for positive
# For binary models, we just use the positive class probability
    raw_score = probs[1].item()  # Probability of positive sentiment

if True:
    # Scale from [0, 1] to [0, 5]
    output = raw_score * 5
else:
    output =  raw_score

In [42]:
train_dataset[0]

{'text': "Best hot & sour soup anywhere I've had in the states. Their Peking eggplant (no pork) with brown rice is pretty yummy too. I've eaten many items but these are the only wow items I've found. \n\nThe last few times I've order take out, the delivery gets worse and worse. Assume 30 more minutes than quoted. Stopped adding silverware, plates, napkins, sauces, etc too. I recommend pickup.",
 'label': 4.0,
 '__index_level_0__': 325352}

In [40]:
output

0.18927499651908875